# Pracovní postup Human-in-the-Loop s Microsoft Agent Framework

## 🎯 Výukové cíle

V tomto zápisníku se naučíte, jak implementovat **human-in-the-loop** pracovní postupy pomocí `RequestInfoExecutor` z Microsoft Agent Framework. Tento výkonný vzor umožňuje pozastavit AI pracovní postupy za účelem získání lidského vstupu, čímž se vaše agenti stanou interaktivními a lidem se dá kontrola nad klíčovými rozhodnutími.

## 🔄 Co je Human-in-the-Loop?

**Human-in-the-loop (HITL)** je návrhový vzor, při kterém AI agenti pozastaví provádění a požádají o lidský vstup před pokračováním. To je zásadní pro:

- ✅ **Kritická rozhodnutí** - Získání lidského schválení před provedením důležitých akcí
- ✅ **Nejednoznačné situace** - Umožnit lidem vyjasnit, když AI není jistá
- ✅ **Preference uživatelů** - Zeptat se uživatelů na výběr mezi několika možnostmi
- ✅ **Soulad a bezpečnost** - Zajistit lidský dohled u regulovaných operací
- ✅ **Interaktivní zážitky** - Vytvářet konverzační agenty, kteří reagují na uživatelský vstup

## 🏗️ Jak to funguje v Microsoft Agent Framework

Framework poskytuje tři klíčové komponenty pro HITL:

1. **`RequestInfoExecutor`** - Speciální vykonávač, který pozastaví pracovní postup a vyvolá `RequestInfoEvent`
2. **`RequestInfoMessage`** - Základní třída pro typizované požadavky posílané lidem
3. **`RequestResponse`** - Koreluje lidské odpovědi s původními požadavky pomocí `request_id`

**Vzor pracovního postupu:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Náš příklad: Rezervace hotelu s potvrzením uživatele

Navážeme na podmíněný pracovní postup přidáním lidského potvrzení **před** navržením alternativních destinací:

1. Uživatel žádá o destinaci (např. "Paříž")
2. `availability_agent` zkontroluje, zda jsou k dispozici pokoje
3. **Pokud nejsou pokoje** → `confirmation_agent` se zeptá "Chcete vidět alternativy?"
4. Pracovní postup se **pozastaví** pomocí `RequestInfoExecutor`
5. **Člověk odpovídá** "ano" nebo "ne" přes konzolový vstup
6. `decision_manager` pokračuje podle odpovědi:
   - **Ano** → Zobrazit alternativní destinace
   - **Ne** → Zrušit žádost o rezervaci
7. Zobrazit konečný výsledek

Tento příklad ukazuje, jak dát uživatelům kontrolu nad návrhy agenta!

---

Pojďme začít! 🚀


## Krok 1: Importovat požadované knihovny

Importujeme standardní komponenty Agent Frameworku plus **specifické třídy pro human-in-the-loop**:
- `RequestInfoExecutor` - Executor, který pozastaví pracovní tok pro lidský vstup
- `RequestInfoEvent` - Událost vyvolaná při požadavku na lidský vstup
- `RequestInfoMessage` - Základní třída pro typované požadavky na data
- `RequestResponse` - Koreluje lidské odpovědi s požadavky
- `WorkflowOutputEvent` - Událost pro detekci výstupů pracovního toku


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Krok 2: Definujte Pydantic modely pro strukturované výstupy

Tyto modely definují **schéma**, které agenti vrátí. Zachováváme všechny modely z podmíněného pracovního postupu a přidáváme:

**Nové pro Human-in-the-Loop:**
- `HumanFeedbackRequest` - podtřída `RequestInfoMessage`, která definuje požadavek odesílaný lidem
  - Obsahuje `prompt` (otázku k položení) a `destination` (kontext ohledně nedostupného města)


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Krok 3: Vytvoření nástroje pro rezervaci hotelu

Stejný nástroj jako v podmíněném pracovním postupu – kontroluje, zda jsou v cílové destinaci k dispozici pokoje.


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## Krok 4: Definujte podmínkové funkce pro směrování

Potřebujeme **čtyři podmínkové funkce** pro náš workflow s člověkem v procesu:

**Ze stávajícího podmínkového workflow:**
1. `has_availability_condition` - Směruje, když jsou hotely K DISPOZICI
2. `no_availability_condition` - Směruje, když hotely NENÍ k dispozici

**Nové pro člověka v procesu:**
3. `user_wants_alternatives_condition` - Směruje, když uživatel říká „ano“ alternativám
4. `user_declines_alternatives_condition` - Směruje, když uživatel říká „ne“ alternativám


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Krok 5: Vytvoření Decision Manager Executor

Toto je **jádro vzoru human-in-the-loop**! `DecisionManager` je vlastní `Executor`, který:

1. **Přijímá lidskou zpětnou vazbu** přes objekty `RequestResponse`
2. **Zpracovává uživatelovo rozhodnutí** (ano/ne)
3. **Řídí pracovní tok** odesíláním zpráv příslušným agentům

Klíčové vlastnosti:
- Používá dekorátor `@handler` k zpřístupnění metod jako kroků pracovního toku
- Přijímá `RequestResponse[HumanFeedbackRequest, str]` obsahující jak původní požadavek, tak uživatelovu odpověď
- Vydává jednoduché zprávy "ano" nebo "ne", které spouštějí naše podmínkové funkce


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## Krok 6: Vytvořte vlastní vykonavatele zobrazení

Stejný vykonavatel zobrazení jako v podmíněném workflow - poskytuje konečné výsledky jako výstup workflow.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Krok 7: Načtení proměnných prostředí

Nakonfigurujte klienta LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## Krok 8: Vytvoření AI agentů a vykonavatelů

Vytvoříme **šest komponent pracovního postupu**:

**Agentů (zabalených v AgentExecutor):**
1. **availability_agent** - Kontroluje dostupnost hotelu pomocí nástroje
2. **confirmation_agent** - 🆕 Připravuje žádost o lidské potvrzení
3. **alternative_agent** - Navrhuje alternativní města (když uživatel řekne ano)
4. **booking_agent** - Povzbuzuje k rezervaci (když jsou dostupné pokoje)
5. **cancellation_agent** - 🆕 Zpracovává zprávu o zrušení (když uživatel řekne ne)

**Speciální vykonavatelé:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, který přeruší pracovní postup pro lidský vstup
7. **decision_manager** - 🆕 Vlastní vykonavatel, který směruje podle lidské odpovědi (již definováno výše)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Krok 9: Vytvoření workflow s člověkem v uzlu

Nyní sestavujeme graf workflow s **podmíněným směrováním** včetně cesty s člověkem v uzlu:

**Struktura workflow:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Klíčové hrany:**
- `availability_agent → confirmation_agent` (když nejsou dostupné pokoje)
- `confirmation_agent → prepare_human_request` (transformace typu)
- `prepare_human_request → request_info_executor` (pauza pro člověka)
- `request_info_executor → decision_manager` (vždy - poskytuje RequestResponse)
- `decision_manager → alternative_agent` (když uživatel řekne "ano")
- `decision_manager → cancellation_agent` (když uživatel řekne "ne")
- `availability_agent → booking_agent` (když jsou pokoje dostupné)
- Všechny cesty končí na `display_result`


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Krok 10: Spusťte testovací případ 1 - Město BEZ dostupnosti (Paříž s lidským potvrzením)

Tento test demonstruje **plný cyklus s lidským zapojením**:

1. Požadavek na hotel v Paříži
2. dostupnostní_agent kontroluje → Žádné pokoje
3. potvrzovací_agent vytváří otázku pro člověka
4. vykonavatel_požadavku_informací **pozastaví tok práce** a vysílá `RequestInfoEvent`
5. **Aplikace detekuje událost a vyzve uživatele v konzoli**
6. Uživatel zadá "ano" nebo "ne"
7. Aplikace odesílá odpověď přes `send_responses_streaming()`
8. decision_manager směruje podle odpovědi
9. Zobrazí se konečný výsledek

**Klíčový vzor:**
- Použijte `workflow.run_stream()` pro první iteraci
- Použijte `workflow.send_responses_streaming(pending_responses)` pro následující iterace
- Sledujte `RequestInfoEvent` pro zjištění, kdy je potřeba lidský vstup
- Sledujte `WorkflowOutputEvent` pro zachycení konečných výsledků


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Krok 11: Spusťte testovací případ 2 - Město S DOSTUPNOSTÍ (Stockholm - Bez potřeby lidského zásahu)

Tento test ukazuje **přímou cestu**, když jsou pokoje dostupné:

1. Požadavek hotelu ve Stockholmu
2. availability_agent kontroluje → Pokoje jsou dostupné ✅
3. booking_agent navrhuje rezervaci
4. display_result zobrazí potvrzení
5. **Není potřeba žádný lidský zásah!**

Pracovní postup úplně obchází cestu s lidským zásahem, pokud jsou pokoje dostupné.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Klíčové poznatky a osvědčené postupy s lidským zapojením

### ✅ Co jste se naučili:

#### 1. **Vzor RequestInfoExecutor**
Vzor lidského zapojení v Microsoft Agent Framework využívá tři klíčové komponenty:
- `RequestInfoExecutor` - Pozastavuje tok práce a vysílá události
- `RequestInfoMessage` - Základní třída pro typované požadavky (zděďte ji!)
- `RequestResponse` - Spojuje lidské odpovědi s původními požadavky

**Důležité pochopení:**
- `RequestInfoExecutor` sám nezískává vstupy - pouze pozastavuje tok práce
- Váš aplikační kód musí naslouchat `RequestInfoEvent` a sbírat vstupy
- Musíte zavolat `send_responses_streaming()` se slovníkem mapujícím `request_id` na odpověď uživatele

#### 2. **Vzor streamovaného provádění**
```python
# První iterace
stream = workflow.run_stream(initial_request)

# Následující iterace (po lidském vstupu)
stream = workflow.send_responses_streaming(pending_responses)

# Vždy zpracovávat události
events = [event async for event in stream]
```

#### 3. **Architektura řízená událostmi**
Naslouchejte konkrétním událostem pro řízení toku práce:
- `RequestInfoEvent` - Je potřeba lidský vstup (tok práce pozastaven)
- `WorkflowOutputEvent` - K dispozici je konečný výsledek (tok práce dokončen)
- `WorkflowStatusEvent` - Změny stavu (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, atd.)

#### 4. **Vlastní vykonavatelé s @handler**
`DecisionManager` ukazuje, jak vytvářet vykonavatele, kteří:
- Používají dekorátor `@handler` k zpřístupnění metod jako kroků toku práce
- Přijímají typované zprávy (např. `RequestResponse[HumanFeedbackRequest, str]`)
- Řídí tok práce zasíláním zpráv dalším vykonavatelům
- Přistupují k kontextu přes `WorkflowContext`

#### 5. **Podmíněné směrování s lidskými rozhodnutími**
Můžete vytvářet podmíněné funkce, které vyhodnocují lidské odpovědi:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Skutečné aplikace:

1. **Schvalovací toky práce**
   - Získat schválení manažera před zpracováním výdajových zpráv
   - Požadovat lidskou kontrolu před odesláním automatizovaných e-mailů
   - Potvrdit vysokohodnotné transakce před provedením

2. **Moderace obsahu**
   - Označit pochybný obsah pro lidskou kontrolu
   - Požádat moderátory o konečné rozhodnutí v okrajových případech
   - Eskalovat na lidi, pokud je AI nejistá

3. **Zákaznický servis**
   - Nechat AI automaticky vyřizovat rutinní dotazy
   - Eskalovat složité problémy na lidské agenty
   - Zeptat se zákazníka, zda chce mluvit s člověkem

4. **Zpracování dat**
   - Požádat lidi o vyřešení nejasných datových záznamů
   - Potvrdit interpretace AI nejasných dokumentů
   - Umožnit uživatelům vybrat mezi více platnými interpretacemi

5. **Systémy kritické pro bezpečnost**
   - Vyžadovat lidské potvrzení před nezvratnými akcemi
   - Získat schválení před přístupem k citlivým datům
   - Potvrdit rozhodnutí v regulovaných odvětvích (zdravotnictví, finance)

6. **Interaktivní agenti**
   - Vytvářet konverzační boty, kteří kladou doplňující otázky
   - Vytvářet průvodce, kteří uživatele navedou složitými procesy
   - Navrhovat agenty, kteří spolupracují s lidmi krok za krokem

### 🔄 Srovnání: S lidským zapojením vs bez něj

| Funkce | Podmíněný tok práce | Tok práce s lidským zapojením |
|---------|---------------------|---------------------------|
| **Provedení** | Jednorázové `workflow.run()` | Smyčka s `run_stream()` + `send_responses_streaming()` |
| **Uživatelský vstup** | Žádný (plně automatizovaný) | Interaktivní výzvy přes `input()` nebo UI |
| **Komponenty** | Agenti + Vykonavatelé | + RequestInfoExecutor + DecisionManager |
| **Události** | Jen AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, atd. |
| **Pozastavení** | Bez pozastavení | Tok práce se pozastaví v RequestInfoExecutor |
| **Lidská kontrola** | Žádná lidská kontrola | Lidé činí klíčová rozhodnutí |
| **Použití** | Automatizace | Spolupráce a dohled |

### 🚀 Pokročilé vzory:

#### Více bodů lidského rozhodnutí
Můžete mít více uzlů `RequestInfoExecutor` ve stejném toku práce:
```python
.add_edge(agent1, request_info_1)  # První lidské rozhodnutí
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Druhé lidské rozhodnutí
.add_edge(decision_manager_2, final_agent)
```

#### Zpracování časových limitů
Implementujte časové limity pro lidské odpovědi:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Výchozí nastavení na bezpečnou možnost
```

#### Bohatá integrace UI
Místo `input()` integrujte s webovým UI, Slackem, Teamsem apod.:
```python
if isinstance(event, RequestInfoEvent):
    # Odeslat oznámení do preferovaného kanálu uživatele
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Podmíněné lidské zapojení
Ptát se na lidský vstup pouze v konkrétních situacích:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Smerujte pouze k člověku, pokud je důvěra nízká nebo hodnota vysoká
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Osvědčené postupy:

1. **Vždy zdědět RequestInfoMessage**
   - Zajišťuje typovou bezpečnost a validaci
   - Umožňuje bohatý kontext pro vykreslování UI
   - Jasně vyjadřuje záměr každého typu požadavku

2. **Používejte popisné výzvy**
   - Zahrňte kontext, co se ptáte
   - Vysvětlete důsledky jednotlivých voleb
   - Držte otázky jednoduché a jasné

3. **Zpracovávejte neočekávaný vstup**
   - Validujte odpovědi uživatelů
   - Poskytněte výchozí hodnoty pro neplatný vstup
   - Poskytujte jasné chybové zprávy

4. **Sledujte ID požadavků**
   - Využijte korelaci mezi request_id a odpověďmi
   - Nesnažte se stav spravovat ručně

5. **Navrhujte pro neblokovací provoz**
   - Neblokujte vlákna čekáním na vstup
   - Používejte asynchronní vzory všude
   - Podporujte souběžné instance toku práce

### 📚 Související koncepty:

- **Agent Middleware** - Zachycování volání agentů (předchozí poznámkový blok)
- **Správa stavu toku práce** - Uchovávání stavu toku práce mezi běhy
- **Spolupráce více agentů** - Kombinovat lidské zapojení s týmy agentů
- **Architektury řízené událostmi** - Budovat reaktivní systémy pomocí událostí

---

### 🎓 Gratulujeme!

Ovládáte toky práce s lidským zapojením v Microsoft Agent Framework! Nyní víte, jak:
- ✅ Pozastavit toky práce pro získání lidského vstupu
- ✅ Použít RequestInfoExecutor a RequestInfoMessage
- ✅ Ovládat streamované provádění pomocí událostí
- ✅ Vytvářet vlastní vykonavatele s @handler
- ✅ Směrovat toky práce na základě lidských rozhodnutí
- ✅ Vytvářet interaktivní AI agenty, kteří spolupracují s lidmi

**Toto je klíčový vzor pro budování důvěryhodných, ovladatelných AI systémů!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
